In [1]:
import os
import sys

# VS Code'un eksik aldığı yolları manuel olarak Windows'a tanıtıyoruz
torch_lib_path = r"C:\Users\Mustafa\s_env\Lib\site-packages\torch\lib"
env_scripts_path = r"C:\Users\Mustafa\s_env\Scripts"

os.environ['PATH'] = torch_lib_path + ";" + env_scripts_path + ";" + os.environ.get('PATH', '')
os.add_dll_directory(torch_lib_path)

import torch
print("PyTorch Başarıyla Yüklendi! Versiyon:", torch.__version__)

PyTorch Başarıyla Yüklendi! Versiyon: 2.13.0+cu126


In [2]:
import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
import mlflow

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("neural_baselines")

TASK_COLS = ["type", "queue", "category", "priority"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

c:\Users\Mustafa\s_env\Lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
c:\Users\Mustafa\s_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [3]:
def light_clean(t):
    if not isinstance(t, str):
        return ""
    t = t.replace("\\n", " ")
    t = re.sub(r"<[^>]+>", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

train_df = pd.read_json("../data/processed/train.jsonl", lines=True)
val_df = pd.read_json("../data/processed/val.jsonl", lines=True)
train_df["body_light_clean"] = train_df["body"].apply(light_clean)
val_df["body_light_clean"] = val_df["body"].apply(light_clean)

In [4]:
encoders = {}
y_train_dict, y_val_dict, class_weights = {}, {}, {}

for col in TASK_COLS:
    le = LabelEncoder()
    y_train_dict[col] = le.fit_transform(train_df[col])
    y_val_dict[col] = le.transform(val_df[col])
    encoders[col] = le

    classes = np.arange(len(le.classes_))
    w = compute_class_weight(class_weight="balanced", classes=classes, y=y_train_dict[col])
    class_weights[col] = torch.tensor(w, dtype=torch.float32).to(device)
    print(col, "-> num_classes:", len(le.classes_))

type -> num_classes: 4
queue -> num_classes: 10
category -> num_classes: 5
priority -> num_classes: 4


In [5]:
def simple_tokenize(text):
    return re.findall(r"\w+", text.lower())

VOCAB_SIZE = 20000
SEQ_LEN = 100

counter = Counter()
for t in train_df["body_light_clean"]:
    counter.update(simple_tokenize(t))

vocab = {"<pad>": 0, "<unk>": 1}
for word, _ in counter.most_common(VOCAB_SIZE - 2):
    vocab[word] = len(vocab)

print("Vocab boyutu:", len(vocab))

def encode(text, vocab, seq_len):
    tokens = simple_tokenize(text)
    ids = [vocab.get(tok, vocab["<unk>"]) for tok in tokens[:seq_len]]
    return ids + [vocab["<pad>"]] * (seq_len - len(ids))

Vocab boyutu: 20000


In [6]:
class TicketDataset(Dataset):
    def __init__(self, texts, y_dict, vocab, seq_len):
        self.texts = texts
        self.y_dict = y_dict
        self.vocab = vocab
        self.seq_len = seq_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids = encode(self.texts[idx], self.vocab, self.seq_len)
        item = {"input_ids": torch.tensor(ids, dtype=torch.long)}
        for col in TASK_COLS:
            item[col] = torch.tensor(self.y_dict[col][idx], dtype=torch.long)
        return item

train_ds = TicketDataset(train_df["body_light_clean"].values, y_train_dict, vocab, SEQ_LEN)
val_ds = TicketDataset(val_df["body_light_clean"].values, y_val_dict, vocab, SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)

In [7]:
# NOT: ModuleDict anahtarlarinda "type" gibi task isimlerini DOGRUDAN kullanma —
# nn.Module'un yerlesik .type() metoduyla catisiyor. "head_" prefix'i zorunlu.

class MultiTaskCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes_dict, kernel_sizes=(3, 4, 5), num_filters=64):
        super().__init__()
        self.task_names = list(num_classes_dict.keys())
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, num_filters, kernel_size=k) for k in kernel_sizes])
        self.dropout = nn.Dropout(0.3)
        self.shared = nn.Linear(num_filters * len(kernel_sizes), 128)
        self.heads = nn.ModuleDict({
            f"head_{task}": nn.Linear(128, n) for task, n in num_classes_dict.items()
        })

    def forward(self, input_ids):
        x = self.embedding(input_ids).permute(0, 2, 1)
        conv_outs = [torch.max(torch.relu(conv(x)), dim=2).values for conv in self.convs]
        x = self.dropout(torch.cat(conv_outs, dim=1))
        shared = self.dropout(torch.relu(self.shared(x)))
        return {task: self.heads[f"head_{task}"](shared) for task in self.task_names}


class MultiTaskRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes_dict):
        super().__init__()
        self.task_names = list(num_classes_dict.keys())
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.shared = nn.Linear(hidden_dim * 2, 128)
        self.heads = nn.ModuleDict({
            f"head_{task}": nn.Linear(128, n) for task, n in num_classes_dict.items()
        })

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        _, (h_n, _) = self.lstm(x)
        h = torch.cat([h_n[0], h_n[1]], dim=1)  # ileri + geri yon son katman
        h = self.dropout(h)
        shared = self.dropout(torch.relu(self.shared(h)))
        return {task: self.heads[f"head_{task}"](shared) for task in self.task_names}

In [8]:
num_classes_dict = {col: len(encoders[col].classes_) for col in TASK_COLS}

def train_model(model, name, epochs=15, patience=3):
    model.to(device)
    criterion = {col: nn.CrossEntropyLoss(weight=class_weights[col]) for col in TASK_COLS}
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_val_loss = float("inf")
    patience_counter = 0
    best_state = None

    with mlflow.start_run(run_name=name):
        mlflow.log_param("architecture", name)
        mlflow.log_param("vocab_size", VOCAB_SIZE)
        mlflow.log_param("seq_len", SEQ_LEN)

        for epoch in range(epochs):
            model.train()
            train_loss = 0
            for batch in train_loader:
                input_ids = batch["input_ids"].to(device)
                optimizer.zero_grad()
                outputs = model(input_ids)
                loss = sum(criterion[col](outputs[col], batch[col].to(device)) for col in TASK_COLS)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
            train_loss /= len(train_loader)

            model.eval()
            val_loss = 0
            correct = {col: 0 for col in TASK_COLS}
            total = 0
            with torch.no_grad():
                for batch in val_loader:
                    input_ids = batch["input_ids"].to(device)
                    outputs = model(input_ids)
                    loss = sum(criterion[col](outputs[col], batch[col].to(device)) for col in TASK_COLS)
                    val_loss += loss.item()
                    total += input_ids.size(0)
                    for col in TASK_COLS:
                        preds = outputs[col].argmax(dim=1).cpu()
                        correct[col] += (preds == batch[col]).sum().item()
            val_loss /= len(val_loader)

            print(f"[{name}] epoch {epoch+1}/{epochs} train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
                  + " ".join(f"{col}_acc={correct[col]/total:.3f}" for col in TASK_COLS))

            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("val_loss", val_loss, step=epoch)
            for col in TASK_COLS:
                mlflow.log_metric(f"val_{col}_accuracy", correct[col] / total, step=epoch)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = model.state_dict()
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping (epoch {epoch+1})")
                    break

        model.load_state_dict(best_state)
        torch.save(model.state_dict(), f"../src/models/{name}.pt")
    return model

cnn_model = MultiTaskCNN(len(vocab), embed_dim=128, num_classes_dict=num_classes_dict)
cnn_model = train_model(cnn_model, "cnn_multitask")

rnn_model = MultiTaskRNN(len(vocab), embed_dim=128, hidden_dim=64, num_classes_dict=num_classes_dict)
rnn_model = train_model(rnn_model, "rnn_multitask")

[cnn_multitask] epoch 1/15 train_loss=5.6536 val_loss=5.0498 type_acc=0.723 queue_acc=0.249 category_acc=0.564 priority_acc=0.340
[cnn_multitask] epoch 2/15 train_loss=4.9034 val_loss=4.7789 type_acc=0.659 queue_acc=0.359 category_acc=0.607 priority_acc=0.459
[cnn_multitask] epoch 3/15 train_loss=4.5928 val_loss=4.5977 type_acc=0.742 queue_acc=0.309 category_acc=0.559 priority_acc=0.387
[cnn_multitask] epoch 4/15 train_loss=4.4018 val_loss=4.5087 type_acc=0.730 queue_acc=0.308 category_acc=0.604 priority_acc=0.440
[cnn_multitask] epoch 5/15 train_loss=4.2204 val_loss=4.4261 type_acc=0.591 queue_acc=0.394 category_acc=0.605 priority_acc=0.419
[cnn_multitask] epoch 6/15 train_loss=4.0591 val_loss=4.3418 type_acc=0.671 queue_acc=0.361 category_acc=0.489 priority_acc=0.403
[cnn_multitask] epoch 7/15 train_loss=3.8936 val_loss=4.2738 type_acc=0.720 queue_acc=0.384 category_acc=0.555 priority_acc=0.440
[cnn_multitask] epoch 8/15 train_loss=3.7216 val_loss=4.2393 type_acc=0.733 queue_acc=0.37